# Alocação dinâmica de memória — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

Este tutorial retoma o programa `agenda.c` visto em aula: a agenda que não tem mais
`MAX_CONTATOS` porque cresce sozinha. Depois de três blocos de prática — um por função —
vêm as tarefas de modificação e o desafio.

## Objetivos

Ao final deste tutorial você será capaz de:

- Alocar um bloco com `malloc(quantidade * sizeof(tipo))`, testar o `NULL` e liberá-lo com `free`;
- Explicar por que o conteúdo de um bloco de `malloc` é indeterminado e quando `calloc` compensa;
- Redimensionar um vetor com `realloc` usando o padrão do ponteiro temporário, sabendo que o
  endereço pode mudar;
- Reconhecer vazamento, ponteiro pendurado, `free` duplo e estouro de bloco — e usar as
  ferramentas que apontam a linha culpada.

In [ ]:
# Confira se o gcc está disponível no seu ambiente
!gcc --version | head -1

## O programa da aula

Recompile e execute o programa condutor **exatamente como saiu da aula** e confira que a saída
bate com a que vimos no slide.

Se a linha `A:` não imprimir `0 0 0 0` na sua máquina, não é bug: é justamente o ponto da
seção 2 deste tutorial.

In [ ]:
%%writefile agenda.c
#include <stdio.h>
#include <stdlib.h>   /* malloc, calloc, realloc, free */
#include <string.h>

int main(void) {
    /* o que vem dentro de um bloco recem-alocado? */
    int *sujo = malloc(4 * sizeof(int));
    for (int i = 0; i < 4; i++)
        sujo[i] = 777;
    free(sujo);
    int *a = malloc(4 * sizeof(int));
    int *b = calloc(4, sizeof(int));
    printf("A: malloc -> %d %d %d %d\n", a[0], a[1], a[2], a[3]);
    printf("B: calloc -> %d %d %d %d\n", b[0], b[1], b[2], b[3]);
    free(a);
    free(b);

    int capacidade = 2, n = 0;
    char **nomes  = malloc(capacidade * sizeof(char *));
    int   *idades = malloc(capacidade * sizeof(int));
    if (nomes == NULL || idades == NULL) {
        printf("sem memoria\n");
        return 1;
    }

    const char *entrada[] = {"ana", "bruno", "carla",
                             "davi", "elis"};
    int idade[] = {23, 19, 31, 24, 28};

    for (int i = 0; i < 5; i++) {
        if (n == capacidade) {
            int nova = capacidade * 2;
            char **n2 = realloc(nomes,  nova * sizeof(char *));
            int   *i2 = realloc(idades, nova * sizeof(int));
            if (n2 == NULL || i2 == NULL) { return 1; }
            printf("C: cheia com %d -> cresce para %d | nomes: %s\n",
                   capacidade, nova,
                   (n2 == nomes) ? "mesmo bloco" : "bloco NOVO");
            nomes = n2;  idades = i2;  capacidade = nova;
        }
        nomes[n] = malloc(strlen(entrada[i]) + 1);
        strcpy(nomes[n], entrada[i]);
        idades[n] = idade[i];
        n++;
    }

    printf("D: %d contatos guardados em capacidade %d\n",
           n, capacidade);
    for (int i = 0; i < n; i++)
        printf("   %-6s %3d\n", nomes[i], idades[i]);
    printf("E: sizeof(nomes) = %zu | strlen(nomes[1]) + 1 = %zu\n",
           sizeof(nomes), strlen(nomes[1]) + 1);

    for (int i = 0; i < n; i++)
        free(nomes[i]);
    free(nomes);
    free(idades);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra agenda.c -o agenda && ./agenda
!./agenda | grep -q 'D: 5 contatos guardados em capacidade 8' && echo OK \
  || echo 'Verifique: esperava 5 contatos em capacidade 8'

## 1. `malloc` e `free`: pedir um bloco e devolvê-lo

`malloc(n)` reserva **n bytes** contíguos na heap e devolve o endereço do primeiro — ou `NULL`
se não houver memória. Ele conta bytes, não elementos: por isso o pedido tem quase sempre a
forma `quantidade * sizeof(tipo)`.

Duas coisas separadas convivem aqui:

- o **ponteiro**, uma variável comum que vive na pilha e morre com a função;
- o **bloco**, que vive na heap e só desaparece quando você chama `free`.

Perder o ponteiro não apaga o bloco — apenas torna impossível devolvê-lo. E `free` não mexe no
seu ponteiro: depois dele, a variável continua com o mesmo endereço, agora inválido.

In [ ]:
%%writefile blocos.c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    int n = 5;
    int *v = malloc(n * sizeof(int));
    if (v == NULL) { printf("sem memoria\n"); return 1; }

    for (int i = 0; i < n; i++)
        v[i] = i * 10;

    printf("sizeof(v)       = %zu bytes  <- o ponteiro, na pilha\n", sizeof(v));
    printf("n * sizeof(int) = %zu bytes <- o bloco, na heap\n", n * sizeof(int));
    printf("v  = %p  (para onde ele aponta)\n", (void *) v);
    printf("&v = %p  (onde o ponteiro mora)\n", (void *) &v);
    printf("v[4] = %d\n", v[4]);

    free(v);
    printf("depois do free, v continua valendo %p\n", (void *) v);
    v = NULL;
    printf("depois de v = NULL, v vale %p\n", (void *) v);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra blocos.c -o blocos && ./blocos

**Repare em três coisas na saída:**

1. `sizeof(v)` é 8 em qualquer alocação: mede o ponteiro, nunca o bloco. Quem quiser saber o
   tamanho do bloco precisa guardá-lo em uma variável à parte — o mesmo `n` que você já
   carregava junto dos vetores.
2. `v` e `&v` são endereços bem distantes: um é heap, o outro é pilha.
3. Depois do `free`, `v` continua apontando para o mesmo lugar. É por isso que `v = NULL`
   logo em seguida é um hábito barato: um ponteiro nulo falha alto e cedo.

In [ ]:
%%writefile ex1.c
#include <stdio.h>
#include <stdlib.h>

/* Exercicio 1: devolva uma COPIA dinamica do vetor recebido.
   TODO: aloque n inteiros, teste o NULL, copie os elementos e
   devolva o endereco. Quem chamar fica responsavel pelo free. */
int *copia_vetor(const int *origem, int n) {
    /* TODO: implemente aqui */
    return NULL;
}

int main(void) {
    int a[5] = {3, 1, 4, 1, 5};
    int *b = copia_vetor(a, 5);
    if (b == NULL) { printf("copia_vetor devolveu NULL\n"); return 1; }

    b[0] = 99;   /* mexer na copia nao pode alterar o original */
    printf("%d %d %d\n", a[0], b[0], b[4]);
    free(b);
    return 0;
}

In [ ]:
!gcc -Wall -g -fsanitize=address ex1.c -o ex1 && ./ex1
!./ex1 | grep -q '^3 99 5$' && echo OK || echo 'Verifique: esperava "3 99 5"'

## 2. `calloc`: o mesmo pedido, com garantia de zeros

O programa da aula encheu um bloco com `777`, liberou e pediu outro do mesmo tamanho — muito
provavelmente o **mesmo bloco de volta**. O que saiu na linha `A:` depende da máquina:

| onde | linha `A:` |
|---|---|
| macOS recente | `0 0 0 0` |
| muitos Linux | `777 777 777 777` |
| com `MallocScribble=1` (macOS) | `-1431655766`, ou seja `0xAAAAAAAA` |

Três respostas para o mesmo código: isso é a definição de **valor indeterminado**. Ler `v[i]`
antes de escrever nele é comportamento indefinido, e zero observado é coincidência, não
contrato.

`calloc(quantidade, tamanho)` resolve isso: aloca `quantidade * tamanho` bytes, **zera tudo** e
ainda detecta o transbordo da multiplicação (devolvendo `NULL` em vez de um bloco pequeno
demais). O preço é o tempo de limpar.

**Regra de bolso:** se a primeira coisa que você faria é um laço escrevendo zeros, use `calloc`.
Se é um laço escrevendo os valores de verdade, use `malloc`.

In [ ]:
# Force a máquina a mostrar que o conteúdo não é garantido.
# No macOS, MallocScribble preenche blocos com 0xAA; no Linux, o próprio
# reaproveitamento do bloco anterior costuma revelar os 777.
!MallocScribble=1 MallocPreScribble=1 ./agenda | head -2
!echo '--- sem a variável de ambiente ---'
!./agenda | head -2

In [ ]:
%%writefile ex2.c
#include <stdio.h>
#include <stdlib.h>

/* Exercicio 2: conte quantas vezes cada letra minuscula aparece em s.
   Devolva um vetor de 26 contadores (indice 0 = 'a', 25 = 'z').
   TODO: escolha entre malloc e calloc -- e saiba justificar a escolha. */
int *conta_letras(const char *s) {
    /* TODO: implemente aqui */
    return NULL;
}

int main(void) {
    int *c = conta_letras("arara azul do sertao");
    if (c == NULL) { printf("conta_letras devolveu NULL\n"); return 1; }

    printf("a=%d o=%d z=%d\n", c['a' - 'a'], c['o' - 'a'], c['z' - 'a']);
    free(c);
    return 0;
}

In [ ]:
!gcc -Wall -g -fsanitize=address ex2.c -o ex2 && ./ex2
!./ex2 | grep -q '^a=5 o=2 z=1$' && echo OK || echo 'Verifique: esperava "a=5 o=2 z=1"'

## 3. `realloc`: mudar de ideia sobre o tamanho

`realloc(p, novos_bytes)` redimensiona o bloco e devolve o endereço do resultado — que **pode
ser outro**. Quando muda de lugar, ele copia o conteúdo antigo e libera o bloco velho por você.
O espaço acrescentado *não* vem zerado.

Daí saem as duas regras da função:

```c
/* ERRADO: se falhar, devolve NULL e o endereco do bloco antigo se perde */
v = realloc(v, nova * sizeof(int));

/* CERTO: so' troca v depois de saber que deu certo */
int *tmp = realloc(v, nova * sizeof(int));
if (tmp == NULL) { /* v ainda vale: da' para se recuperar */ }
v = tmp;
```

E, depois da troca, existe **um** ponteiro válido: o que `realloc` devolveu. O antigo está
pendurado.

In [ ]:
%%writefile crescer.c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    int capacidade = 2, n = 0;
    int *v = malloc(capacidade * sizeof(int));
    if (v == NULL) return 1;

    for (int i = 1; i <= 9; i++) {
        if (n == capacidade) {
            int nova = capacidade * 2;
            int *tmp = realloc(v, nova * sizeof(int));
            if (tmp == NULL) { free(v); return 1; }
            printf("cresce %d -> %2d | antes %p | depois %p | %s\n",
                   capacidade, nova, (void *) v, (void *) tmp,
                   (tmp == v) ? "mesmo bloco" : "bloco NOVO");
            v = tmp;
            capacidade = nova;
        }
        v[n++] = i;
    }

    printf("v[0] = %d e v[%d] = %d: o conteudo veio junto\n", v[0], n - 1, v[n - 1]);
    free(v);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra crescer.c -o crescer && ./crescer
!echo '--- rode de novo: os endereços mudam, o padrão não ---'
!./crescer

Repare que às vezes sai `mesmo bloco` e às vezes `bloco NOVO`: quando há espaço livre logo
depois do bloco, o alocador apenas o estende. Você não controla isso — e é por isso que nunca
se deve guardar o endereço antigo.

**Por que dobrar a capacidade, e não somar 1?** Crescendo de 1 em 1 até $n$, o total de
elementos copiados é da ordem de $n^2$. Dobrando, fica da ordem de $n$: cada elemento é movido,
em média, um número constante de vezes. A tarefa 1 mede isso na prática.

In [ ]:
%%writefile ex3.c
#include <stdio.h>
#include <stdlib.h>

/* Exercicio 3: acrescente um valor ao fim de um vetor dinamico,
   dobrando a capacidade quando ela se esgotar.
   TODO: complete o crescimento usando o padrao do ponteiro temporario.
   A funcao devolve o endereco do vetor (que pode ter mudado!) e
   atualiza n e capacidade atraves dos ponteiros. */
int *acrescenta(int *v, int *n, int *capacidade, int valor) {
    if (*n == *capacidade) {
        /* TODO: dobre a capacidade com realloc */
    }
    /* TODO: guarde valor na posicao *n e incremente *n */
    return v;
}

int main(void) {
    int capacidade = 1, n = 0;
    int *v = malloc(capacidade * sizeof(int));
    if (v == NULL) return 1;

    for (int i = 1; i <= 10; i++)
        v = acrescenta(v, &n, &capacidade, i * i);

    if (n == 0) { printf("nada foi acrescentado ainda\n"); free(v); return 1; }

    int soma = 0;
    for (int i = 0; i < n; i++) soma += v[i];
    printf("n=%d capacidade=%d soma=%d ultimo=%d\n",
           n, capacidade, soma, v[n - 1]);
    free(v);
    return 0;
}

In [ ]:
!gcc -Wall -g -fsanitize=address ex3.c -o ex3 && ./ex3
!./ex3 | grep -q '^n=10 capacidade=16 soma=385 ultimo=100$' && echo OK \
  || echo 'Verifique: esperava "n=10 capacidade=16 soma=385 ultimo=100"'

## 4. As quatro falhas clássicas

| Falha | O que é | Sintoma típico |
|---|---|---|
| Vazamento | alocar e nunca liberar | o programa incha com o tempo |
| Ponteiro pendurado | usar depois do `free` | lixo ou *segfault* aleatório |
| `free` duplo | liberar duas vezes | aborta dentro de `free`, sem pista |
| Estouro do bloco | escrever além do tamanho pedido | corrompe o vizinho |

O que as quatro têm em comum: **nenhuma falha onde foi cometida**. O programa quebra depois, em
outro lugar, com uma mensagem que aponta para código inocente. Daí a importância das
ferramentas.

In [ ]:
%%writefile vaza.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

void carrega_agenda(void) {
    const char *entrada[] = {"ana", "bruno", "carla", "davi", "elis"};
    char **nomes = malloc(5 * sizeof(char *));
    for (int i = 0; i < 5; i++) {
        nomes[i] = malloc(strlen(entrada[i]) + 1);
        strcpy(nomes[i], entrada[i]);
    }
    printf("agenda com 5 nomes: %s ... %s\n", nomes[0], nomes[4]);
    free(nomes);   /* liberou o vetor de ponteiros, mas nao os nomes */
}

int main(void) { carrega_agenda(); return 0; }

In [ ]:
# A agenda liberou o vetor de ponteiros, mas não os cinco nomes.
!gcc -Wall -Wextra -g vaza.c -o vaza && ./vaza

# macOS: o detector aponta a FUNÇÃO onde cada bloco perdido foi alocado
!MallocStackLogging=1 leaks -atExit -- ./vaza 2>/dev/null | grep -E 'leaks for|ROOT LEAK' | head -3

# Linux: use uma das duas linhas abaixo
# !gcc -g -fsanitize=address vaza.c -o vaza && ./vaza
# !valgrind --leak-check=full ./vaza

In [ ]:
%%writefile uaf.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int main(void) {
    char *nome = malloc(6);
    strcpy(nome, "carla");
    free(nome);
    printf("depois do free: %s\n", nome);   /* ponteiro pendurado */
    return 0;
}

In [ ]:
# Sem o sanitizer isto costuma "funcionar" -- e é esse o problema.
!gcc -Wall -g uaf.c -o uaf_sem && ./uaf_sem
!echo '=== agora com -fsanitize=address ==='
!gcc -Wall -g -fsanitize=address uaf.c -o uaf && ./uaf 2>&1 | head -8

Leia a mensagem inteira do sanitizer: ela diz **o que** (`heap-use-after-free`), **onde**
(a linha do `printf`) e **por causa de quem** (`freed by thread T0 here:` seguido da linha do
`free`). Um *segfault* comum diria apenas "o programa morreu".

### Higiene: cinco hábitos

1. pedir `quantidade * sizeof(tipo)`, nunca um número solto;
2. testar `NULL` logo depois de alocar;
3. receber `realloc` em um ponteiro temporário;
4. liberar de dentro para fora, e uma vez só;
5. `p = NULL` depois do `free`, e compilar com `-Wall -g -fsanitize=address` enquanto
   desenvolve.

## Tarefas

Trabalhe sobre uma cópia do programa da aula. A célula abaixo cria `agenda_tarefa.c` a partir
de `agenda.c`; edite esse arquivo (pelo Jupyter ou pelo seu editor) e recompile.

**Tarefa 1.** Troque a capacidade inicial de 2 para 1 e o fator de crescimento de $\times 2$
para $+1$. Quantas vezes a linha `C:` passa a aparecer? Explique o número — e diga quantos
elementos foram copiados ao todo em cada uma das duas estratégias.

**Tarefa 2.** Troque o `malloc` de `idades` por `calloc` e imprima `idades[7]` logo depois de
alocar, antes de preencher. O que muda em relação ao `malloc`? Rode também com
`MallocScribble=1` para ver a diferença aparecer.

**Tarefa 3.** Comente o laço que libera `nomes[i]` e rode o detector de vazamentos. Quantos
blocos ele acusa? Por que esse número, e não 1 nem 8?

In [ ]:
!cp agenda.c agenda_tarefa.c
# edite agenda_tarefa.c e depois rode esta célula de novo
!gcc -Wall -Wextra agenda_tarefa.c -o agenda_tarefa && ./agenda_tarefa

## Desafio: uma agenda que também remove

Escreva `remover(char **nomes, int *idades, int n, const char *quem)`, que apaga um contato e
devolve a nova quantidade.

Requisitos:

- libere o nome removido **antes** de perder o endereço dele;
- feche o buraco deslocando os contatos seguintes;
- se a agenda ficar com menos de um quarto da capacidade, encolha o vetor pela metade com
  `realloc` (lembre-se: o endereço pode mudar, então a capacidade e os vetores precisam voltar
  para quem chamou);
- remover alguém que não está na agenda deve devolver `n` sem alterar nada;
- o programa deve terminar sem vazamentos e sem nenhum erro do sanitizer.

In [ ]:
%%writefile desafio.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

/* TODO: devolve a nova quantidade de contatos.
   Dica: para encolher o vetor voce vai precisar alterar os ponteiros de
   quem chamou -- pense em como passar 'nomes' e 'idades' por referencia. */
int remover(char **nomes, int *idades, int n, const char *quem) {
    /* TODO: implemente aqui */
    return n;
}

int main(void) {
    /* TODO: monte a agenda, remova "carla" e um nome inexistente,
       imprima o resultado e libere tudo. */
    return 0;
}

In [ ]:
!gcc -Wall -g -fsanitize=address desafio.c -o desafio && ./desafio
!MallocStackLogging=1 leaks -atExit -- ./desafio 2>/dev/null | grep -E 'leaks for' | head -1

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. Para esta aula, os dois pontos de
partida são o capítulo 7.8.5 de Kernighan & Ritchie e o capítulo 12 de Backes.